# 🕸️ DeepFetch
**Universal web scraping & automation engine — stealth Playwright, AI extraction, REST API.**

**Run Cell 1 once** (installs system deps — ~5 min).  
**Run Cell 2 every time** to start or restart the server.

Your public dashboard link appears at the end of Cell 2.


In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 1 — System dependencies (run once per Colab session)
# ═══════════════════════════════════════════════════════════
import subprocess, os

def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print((r.stdout + r.stderr)[-3000:])
        raise RuntimeError(f'Failed: {cmd}')
    return r.stdout.strip()

# Node.js 22
print('📦 Node.js 22 ...')
sh('curl -fsSL https://deb.nodesource.com/setup_22.x | bash - 2>&1 | tail -3')
sh('apt-get install -y nodejs 2>&1 | tail -3')
print(f'   ✅ Node {sh("node --version")}  npm {sh("npm --version")}')

# Build tools — required by better-sqlite3 (native C++ addon)
print('📦 Build tools (gcc, python3, node-gyp) ...')
sh('apt-get install -y --no-install-recommends '
   'build-essential python3 python3-dev make g++ '
   '2>&1 | tail -3')
print('   ✅ done')

# Playwright system libs
print('📦 Playwright system libs ...')
sh('apt-get install -y --no-install-recommends '
   'libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 '
   'libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 '
   'libgbm1 libasound2 libpango-1.0-0 libpangocairo-1.0-0 '
   '2>&1 | tail -3')
print('   ✅ done')

# cloudflared
print('📦 cloudflared ...')
sh('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/'
   'cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && '
   'chmod +x /usr/local/bin/cloudflared')
print(f'   ✅ {sh("cloudflared --version").split()[2]}')

# Bump npm to latest stable (avoids "Exit handler never called" bug)
print('📦 Updating npm ...')
sh('npm install -g npm@latest 2>&1 | tail -3')
print(f'   ✅ npm {sh("npm --version")}')

print()
print('🎉 System ready — now run Cell 2.')


In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 2 — Clone · Build · Launch · Expose
#  Safe to re-run: skips steps that are already done.
#  Your dashboard link appears at the bottom.
# ═══════════════════════════════════════════════════════════
import subprocess, os, time, re, secrets

def sh(cmd, cwd=None, show_tail=0):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if r.returncode != 0:
        print((r.stdout + r.stderr)[-4000:])
        raise RuntimeError(f'Failed ({r.returncode}): {cmd}')
    if show_tail:
        lines = (r.stdout + r.stderr).splitlines()
        if lines: print('\n'.join(lines[-show_tail:]))
    return r.stdout.strip()

REPO  = '/deepfetch'
SRV   = f'{REPO}/server'
DASH  = f'{REPO}/dashboard'
DIST  = f'{SRV}/dist/index.js'
TSC   = f'{SRV}/node_modules/.bin/tsc'
PW    = '/ms-playwright'
PORT  = 3000
DATA  = '/deepfetch-data'
CFG   = f'{REPO}/config.yaml'

os.makedirs(DATA, exist_ok=True)
os.environ['PLAYWRIGHT_BROWSERS_PATH'] = PW

# Kill leftovers
subprocess.run("pkill -f 'node.*dist/index' 2>/dev/null; pkill -f cloudflared 2>/dev/null",
               shell=True, capture_output=True)
time.sleep(1)

# ── 1. Clone / pull ──────────────────────────────────────
if not os.path.exists(REPO):
    print('📥 Cloning ...')
    sh(f'git clone --depth 1 https://github.com/ferelking242/deepfetch {REPO}')
else:
    print('📥 Pulling latest ...')
    sh('git pull --ff-only', cwd=REPO)

# ── 2. npm install ────────────────────────────────────────
NPM_FLAGS = '--no-audit --no-fund --prefer-offline'

if not os.path.exists(f'{SRV}/node_modules/.bin/tsc'):
    print('📦 npm install (server) — compiling native modules, takes ~2 min ...')
    sh(f'npm install {NPM_FLAGS} 2>&1', cwd=SRV, show_tail=5)
else:
    print('📦 server node_modules OK')

if not os.path.exists(f'{DASH}/node_modules'):
    print('📦 npm install (dashboard) ...')
    sh(f'npm install {NPM_FLAGS} 2>&1', cwd=DASH, show_tail=3)
else:
    print('📦 dashboard node_modules OK')

# ── 3. Playwright Chromium ────────────────────────────────
has_pw = os.path.exists(PW) and any(
    d.startswith('chromium') for d in os.listdir(PW)) if os.path.exists(PW) else False
if not has_pw:
    print('🎭 Installing Playwright Chromium ...')
    sh(f'node {SRV}/node_modules/.bin/playwright install chromium --with-deps 2>&1',
       show_tail=5)
    print('   ✅ Playwright ready')
else:
    print('🎭 Playwright Chromium OK')

# ── 4. Build TypeScript ───────────────────────────────────
if not os.path.exists(TSC):
    raise RuntimeError(f'TypeScript compiler not found at {TSC}\n'
                       'Re-run Cell 1, then re-run this cell.')

if not os.path.exists(DIST):
    print('🔨 Building TypeScript server ...')
    sh(f'node {TSC} --project tsconfig.json 2>&1', cwd=SRV, show_tail=10)
    if not os.path.exists(DIST):
        raise RuntimeError(f'{DIST} still missing after build')
    print('   ✅ Server built')
else:
    print('🔨 Server dist OK')

# ── 5. Build dashboard ───────────────────────────────────
VITE = f'{DASH}/node_modules/.bin/vite'
if not os.path.exists(f'{DASH}/dist/index.html'):
    print('🔨 Building dashboard ...')
    sh(f'node {VITE} build 2>&1', cwd=DASH, show_tail=5)
    print('   ✅ Dashboard built')
else:
    print('🔨 Dashboard dist OK')

# ── 6. Config ─────────────────────────────────────────────
if not os.path.exists(CFG):
    print('⚙️  Writing config.yaml ...')
    try:
        import yaml
    except ImportError:
        sh('pip install pyyaml -q')
        import yaml
    cfg = {
        'server':    {'port': PORT, 'host': '0.0.0.0',
                      'master_secret': secrets.token_hex(32)},
        'browser':   {'pool_max': 0, 'pool_reserved': 1,
                      'context_ttl_seconds': 300,
                      'navigation_timeout_ms': 30000, 'headless': True},
        'resources': {'cpu_threshold_pct': 85, 'ram_threshold_pct': 80},
        'queue':     {'max_retries': 3, 'retry_base_delay_ms': 2000,
                      'result_ttl_seconds': 86400},
        'ai_engine': {'enabled': True, 'trigger': 'on_selector_failure',
                      'max_html_chars': 50000, 'timeout_ms': 15000,
                      'providers': [
                          {'name': 'ollama', 'local': True, 'model': 'llama3.2',
                           'base_url': 'http://localhost:11434'},
                          {'name': 'groq',   'api_key': '', 'model': 'llama-3.3-70b-versatile'},
                          {'name': 'gemini', 'api_key': '', 'model': 'gemini-2.0-flash'},
                          {'name': 'openai', 'api_key': '', 'model': 'gpt-4o-mini'},
                      ]},
        'zeusdl':    {'binary': 'zeusdl', 'extra_flags': []},
        'sessions':  {'encryption_key': secrets.token_hex(32),
                      'check_interval_seconds': 1800},
        'data_dir':  DATA,
    }
    with open(CFG, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print('   ✅ config.yaml written')
else:
    print('⚙️  config.yaml exists — keeping it')

# ── 7. Start server ───────────────────────────────────────
print('🚀 Starting DeepFetch server ...')
srv_log = open('/tmp/deepfetch.log', 'w')
subprocess.Popen(
    ['node', DIST],
    env={**os.environ, 'DF_CONFIG': CFG, 'NODE_ENV': 'production'},
    stdout=srv_log, stderr=srv_log, cwd=REPO,
)

import urllib.request
for i in range(40):
    time.sleep(1)
    try:
        urllib.request.urlopen(f'http://localhost:{PORT}/v1/health', timeout=2)
        print(f'   ✅ Server up in {i+1}s')
        break
    except:
        pass
else:
    print('❌ Server failed. Logs:')
    print(open('/tmp/deepfetch.log').read()[-4000:])
    raise RuntimeError('DeepFetch did not start')

# ── 8. cloudflared tunnel ─────────────────────────────────
print('🌐 Opening public tunnel ...')
cf_log = open('/tmp/cf.log', 'w')
subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=cf_log, stderr=subprocess.STDOUT,
)

PUBLIC_URL = None
for _ in range(30):
    time.sleep(1)
    try:
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com',
                      open('/tmp/cf.log').read())
        if m:
            PUBLIC_URL = m.group(0)
            break
    except:
        pass

if not PUBLIC_URL:
    PUBLIC_URL = f'http://localhost:{PORT}'
    print('⚠️  cloudflared URL not found')

print()
print('═' * 56)
print('  🎉  DeepFetch is LIVE')
print('═' * 56)
print(f'  📊  Dashboard : {PUBLIC_URL}/dashboard')
print(f'  📖  API Docs  : {PUBLIC_URL}/docs')
print(f'  ❤️   Health    : {PUBLIC_URL}/v1/health')
print('═' * 56)
print('  Open Dashboard on any device (phone, PC, tablet)')
print('  Configure AI keys and sessions directly there.')
